###**Dynamic Capabilities**

In [0]:
dbutils.widgets.text('file_name','orders')

In [0]:
file_name = dbutils.widgets.get('file_name')

In [0]:
df = spark.read.format('parquet').load(f'abfss://source@etehub.dfs.core.windows.net/{file_name}')

df.display()

### **Data Reading**

In [0]:
stream_df = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format','parquet')\
    .option('cloudFiles.schemaLocation',f'abfss://bronze@etehub.dfs.core.windows.net/checkpoint_loc_{file_name}')\
    .load(f'abfss://source@etehub.dfs.core.windows.net/{file_name}')

###**Data Writing**

In [0]:
stream_df.writeStream.format('delta')\
    .outputMode('append')\
    .option('checkpointLocation',f'abfss://bronze@etehub.dfs.core.windows.net/checkpoint_loc_{file_name}')\
    .option('path',f'abfss://bronze@etehub.dfs.core.windows.net/{file_name}')\
    .trigger(once=True)\
    .start()
